In [27]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

# Modeling + preprocessing (we'll use these later)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
)
import warnings
warnings.filterwarnings("ignore")

# 1. Load the dataset
df = pd.read_csv("medical_insurance.csv")

# 2. Take a quick look at the data
print("Shape of data:", df.shape)
display(df.head())

# 3. Check the target distribution for is_high_risk
print("\n'is_high_risk' value counts:")
print(df["is_high_risk"].value_counts())

print("\n'is_high_risk' proportion:")
print(df["is_high_risk"].value_counts(normalize=True))

Shape of data: (100000, 54)


,person_id,age,sex,region,urban_rural,income,education,marital_status,employment_status,household_size,...,liver_disease,arthritis,mental_health,proc_imaging_count,proc_surgery_count,proc_physio_count,proc_consult_count,proc_lab_count,is_high_risk,had_major_procedure
0,75722,52,Female,North,Suburban,22700.0,Doctorate,Married,Retired,3,...,0,1,0,1,0,2,0,1,0,0
1,80185,79,Female,North,Urban,12800.0,No HS,Married,Employed,3,...,0,1,1,0,0,1,0,1,1,0
2,19865,68,Male,North,Rural,40700.0,HS,Married,Retired,5,...,0,0,1,1,0,2,1,0,1,0
3,76700,15,Male,North,Suburban,15600.0,Some College,Married,Self-employed,5,...,0,0,0,1,0,0,1,0,0,0
4,92992,53,Male,Central,Suburban,89600.0,Doctorate,Married,Self-employed,2,...,0,1,0,2,0,1,1,0,1,0



'is_high_risk' value counts:
is_high_risk
0    63219
1    36781
Name: count, dtype: int64

'is_high_risk' proportion:
is_high_risk
0    0.63219
1    0.36781
Name: proportion, dtype: float64


In [2]:
''' Build Feature Lists Automatically '''
# 1. Identify all columns in the dataset
all_cols = df.columns.tolist()

# 2. Columns we DO NOT want as inputs for the risk model
cols_to_drop = [
    "person_id",        # ID column
    "is_high_risk",     # TARGET variable - must be excluded
    "annual_premium",   # second target, not for risk modeling
    "risk_score",       # not created yet but ensure excluded
    "risk_band",
    "monthly_premium"
]

# 3. Keep all other columns as potential features
feature_cols = [c for c in all_cols if c not in cols_to_drop]

# 4. Auto-detect NUMERIC columns (int or float)
numeric_cols = df[feature_cols].select_dtypes(include=["int64", "float64"]).columns.tolist()

# 5. Detect BINARY columns (0/1 only inside numeric group)
binary_cols = [
    c for c in numeric_cols
    if df[c].nunique() == 2 and set(df[c].unique()) <= {0, 1}
]

# 6. Remove binary columns from numeric list to leave TRUE numeric values
numeric_cols = [c for c in numeric_cols if c not in binary_cols]

# 7. Detect CATEGORICAL columns (dtype = object)
categorical_cols = df[feature_cols].select_dtypes(include=["object"]).columns.tolist()

# 8. Show results
print("Total features for modeling:", len(feature_cols))
print("\nNumeric columns:", numeric_cols)
print("\nBinary columns:", binary_cols)
print("\nCategorical columns:", categorical_cols)

Total features for modeling: 49

Numeric columns: ['age', 'income', 'household_size', 'dependents', 'bmi', 'visits_last_year', 'hospitalizations_last_3yrs', 'days_hospitalized_last_3yrs', 'medication_count', 'systolic_bp', 'diastolic_bp', 'ldl', 'hba1c', 'deductible', 'copay', 'policy_term_years', 'policy_changes_last_2yrs', 'provider_quality', 'annual_medical_cost', 'claims_count', 'avg_claim_amount', 'total_claims_paid', 'chronic_count', 'proc_imaging_count', 'proc_surgery_count', 'proc_physio_count', 'proc_consult_count', 'proc_lab_count']

Binary columns: ['hypertension', 'diabetes', 'asthma', 'copd', 'cardiovascular_disease', 'cancer_history', 'kidney_disease', 'liver_disease', 'arthritis', 'mental_health', 'had_major_procedure']

Categorical columns: ['sex', 'region', 'urban_rural', 'education', 'marital_status', 'employment_status', 'smoker', 'alcohol_freq', 'plan_type', 'network_tier']


In [3]:
''' Train/Test Split + Preprocessing Pipeline '''
# 1. Define X and y
X = df[feature_cols]              # all predictor variables
y = df["is_high_risk"]            # target for risk prediction

# 2. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y        # important for classification
)

# 3. Build preprocessing pipeline
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),                     # scale continuous vars
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),  # encode string vars
        ("bin", "passthrough", binary_cols)                          # leave binary vars unchanged
    ]
)

# Quick check
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Preprocessing pipeline ready.")

Train shape: (80000, 49)
Test shape: (20000, 49)
Preprocessing pipeline ready.


In [4]:
''' Logistic Regression (Baseline) '''
# Build full pipeline: preprocessing + model
baseline_logistic = Pipeline([
    ("preprocess", preprocess),
    ("clf", LogisticRegression(max_iter=500))
])

# Train the model
baseline_logistic.fit(X_train, y_train)

# Predictions
pred_proba = baseline_logistic.predict_proba(X_test)[:, 1]   # probability of high risk
pred = baseline_logistic.predict(X_test)                      # class 0/1 prediction

# Evaluation
print("Accuracy:", accuracy_score(y_test, pred))
print("F1 Score:", f1_score(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, pred_proba))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, pred))
print("\nClassification Report:\n", classification_report(y_test, pred))

Accuracy: 0.9723
F1 Score: 0.9623334239869459
ROC-AUC: 0.9976506641316932

Confusion Matrix:
 [[12369   275]
 [  279  7077]]

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.98      0.98     12644
           1       0.96      0.96      0.96      7356

    accuracy                           0.97     20000
   macro avg       0.97      0.97      0.97     20000
weighted avg       0.97      0.97      0.97     20000



In [5]:
''' Regularized Logistic Regression Models '''
from sklearn.model_selection import GridSearchCV

# 1. Ridge Logistic Regression (L2 Regularization)
ridge_pipe = Pipeline([
    ("preprocess", preprocess),
    ("clf", LogisticRegression(penalty="l2", max_iter=1000))
])

ridge_params = {
    "clf__C": [0.01, 0.1, 1, 10]      # smaller C = stronger regularization
}

grid_ridge = GridSearchCV(
    ridge_pipe,
    ridge_params,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_ridge.fit(X_train, y_train)

print("Best Ridge C:", grid_ridge.best_params_)
print("Best Ridge ROC-AUC:", grid_ridge.best_score_)

# 2. Lasso Logistic Regression (L1 Regularization)
lasso_pipe = Pipeline([
    ("preprocess", preprocess),
    ("clf", LogisticRegression(
        penalty="l1",
        solver="liblinear",
        max_iter=1000
    ))
])

lasso_params = {
    "clf__C": [0.01, 0.1, 1, 10]
}

grid_lasso = GridSearchCV(
    lasso_pipe,
    lasso_params,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_lasso.fit(X_train, y_train)

print("\nBest Lasso C:", grid_lasso.best_params_)
print("Best Lasso ROC-AUC:", grid_lasso.best_score_)

# 3. Elastic Net Logistic Regression (L1 + L2)
enet_pipe = Pipeline([
    ("preprocess", preprocess),
    ("clf", LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        max_iter=2000
    ))
])

enet_params = {
    "clf__C": [0.01, 0.1, 1],
    "clf__l1_ratio": [0.1, 0.5, 0.9]   # mix of L1 and L2
}

grid_enet = GridSearchCV(
    enet_pipe,
    enet_params,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_enet.fit(X_train, y_train)

print("\nBest ElasticNet Params:", grid_enet.best_params_)
print("Best ElasticNet ROC-AUC:", grid_enet.best_score_)

Best Ridge C: {'clf__C': 10}
Best Ridge ROC-AUC: 0.9972182971444363

Best Lasso C: {'clf__C': 0.1}
Best Lasso ROC-AUC: 0.9972412851191754

Best ElasticNet Params: {'clf__C': 0.1, 'clf__l1_ratio': 0.9}
Best ElasticNet ROC-AUC: 0.9972393296965943


In [6]:
''' Train Final Lasso Model + Generate Risk Score '''
# 1. Retrieve the best Lasso model from grid
best_lasso_model = grid_lasso.best_estimator_

# Fit on entire training set (already done inside GridSearchCV)
# but we explicitly refit again to confirm
best_lasso_model.fit(X_train, y_train)

# 2. Generate risk_score (Predicted Probability)
df["risk_score"] = best_lasso_model.predict_proba(
    df[feature_cols]
)[:, 1]

# 3. Create 3 risk bands (Low / Medium / High)
df["risk_band"] = pd.cut(
    df["risk_score"],
    bins=[0, 0.3, 0.7, 1.0],
    labels=["Low", "Medium", "High"]
)

# 4. Inspect
print(df[["risk_score", "risk_band"]].head())

print("\nRisk Band Distribution:")
print(df["risk_band"].value_counts())

     risk_score risk_band
0  4.457732e-01    Medium
1  1.000000e+00      High
2  1.000000e+00      High
3  7.761985e-11       Low
4  9.999990e-01      High

Risk Band Distribution:
risk_band
Low       61451
High      34886
Medium     3663
Name: count, dtype: int64


In [7]:
''' Prepare Features for Premium Model '''
# 1. Define target for premium model
y2 = df["annual_premium"]

# 2. Define features (include risk_score)
premium_feature_cols = feature_cols + ["risk_score"]  # add risk score into features

# Also add encoded version of risk_band
premium_feature_cols.append("risk_band")

# 3. Auto-detect types again
# Numeric
numeric_cols_2 = df[premium_feature_cols].select_dtypes(include=["int64", "float64"]).columns.tolist()

# Binary (still 0/1)
binary_cols_2 = [
    c for c in numeric_cols_2
    if df[c].nunique() == 2 and set(df[c].unique()) <= {0, 1}
]

# Remove binary from numeric
numeric_cols_2 = [c for c in numeric_cols_2 if c not in binary_cols_2]

# Categorical (object dtype including risk_band)
categorical_cols_2 = df[premium_feature_cols].select_dtypes(include=["object"]).columns.tolist()

# 4. Prepare X
X2 = df[premium_feature_cols]

# 5. Train-test split for premium model
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2,
    test_size=0.2,
    random_state=42
)

# 6. Preprocessor for premium model
preprocess_2 = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols_2),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols_2),
        ("bin", "passthrough", binary_cols_2)
    ]
)

print("Premium model feature count:", len(premium_feature_cols))
print("Numeric:", len(numeric_cols_2))
print("Binary:", len(binary_cols_2))
print("Categorical:", len(categorical_cols_2))

Premium model feature count: 51
Numeric: 29
Binary: 11
Categorical: 10


In [19]:
# Baseline Linear Regression (OLS)
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Build pipeline
baseline_lr = Pipeline([
    ("preprocess", preprocess_2),
    ("reg", LinearRegression())
])

# Fit model
baseline_lr.fit(X2_train, y2_train)

# Predict
pred_lr = baseline_lr.predict(X2_test)

# Evaluation
mse_lr = mean_squared_error(y2_test, pred_lr)
rmse_lr = np.sqrt(mse_lr)
mae_lr = mean_absolute_error(y2_test, pred_lr)
r2_lr = r2_score(y2_test, pred_lr)

print("Linear Regression Performance:")
print("RMSE:", rmse_lr)
print("MAE:", mae_lr)
print("R²:", r2_lr)

Linear Regression Performance:
RMSE: 74.44710254500586
MAE: 37.66101153659216
R²: 0.965173681306704


In [35]:
''' Regularized Regression '''
from sklearn.linear_model import Ridge, Lasso, ElasticNet

# Ridge Regression
ridge_pipe = Pipeline([
    ("preprocess", preprocess_2),
    ("reg", Ridge())
])

ridge_params = {
    "reg__alpha": [0.001, 0.01, 0.1, 1, 10]
}

grid_ridge_reg = GridSearchCV(
    ridge_pipe,
    ridge_params,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

grid_ridge_reg.fit(X2_train, y2_train)

ridge_mse = -grid_ridge_reg.best_score_
ridge_rmse = np.sqrt(ridge_mse)

print("Best Ridge alpha:", grid_ridge_reg.best_params_)
print("Ridge RMSE:", ridge_rmse)

# Lasso Regression
lasso_pipe = Pipeline([
    ("preprocess", preprocess_2),
    ("reg", Lasso(
        max_iter=20000,
        tol=1e-4,
        selection="cyclic",
        fit_intercept=True
    ))
])

lasso_params = {
    "reg__alpha": [1, 10, 50, 100]
}

grid_lasso_reg = GridSearchCV(
    lasso_pipe,
    lasso_params,
    cv=3,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

grid_lasso_reg.fit(X2_train, y2_train)

lasso_rmse = np.sqrt(-grid_lasso_reg.best_score_)

print("\nBest Lasso alpha:", grid_lasso_reg.best_params_)
print("Lasso RMSE:", lasso_rmse)

# Evaluate on test set if you want
pred_lasso = grid_lasso_reg.best_estimator_.predict(X2_test)
r2_lasso = r2_score(y2_test, pred_lasso)
print("Lasso R² (test):", r2_lasso)

# Elastic Net Regression
enet_pipe = Pipeline([
    ("preprocess", preprocess_2),
    ("reg", ElasticNet(
        max_iter=30000,
        tol=1e-4,
        selection="cyclic",
        fit_intercept=True
    ))
])

enet_params = {
    "reg__alpha": [1, 10, 50, 100],
    "reg__l1_ratio": [0.1, 0.5, 0.9]
}

grid_enet_reg = GridSearchCV(
    enet_pipe,
    enet_params,
    cv=3,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

grid_enet_reg.fit(X2_train, y2_train)

enet_rmse = np.sqrt(-grid_enet_reg.best_score_)

print("\nBest ElasticNet params:", grid_enet_reg.best_params_)
print("ElasticNet RMSE:", enet_rmse)

pred_enet = grid_enet_reg.best_estimator_.predict(X2_test)
r2_enet = r2_score(y2_test, pred_enet)
print("ElasticNet R² (test):", r2_enet)

Best Ridge alpha: {'reg__alpha': 10}
Ridge RMSE: 75.23213095274164

Best Lasso alpha: {'reg__alpha': 1}
Lasso RMSE: 75.37702001152826
Lasso R² (test): 0.9651658282470784

Best ElasticNet params: {'reg__alpha': 1, 'reg__l1_ratio': 0.9}
ElasticNet RMSE: 92.66851545214294
ElasticNet R² (test): 0.945043181832711


In [37]:
''' Random Forest Regressor '''
from sklearn.ensemble import RandomForestRegressor

rf_reg = Pipeline([
    ("preprocess", preprocess_2),
    ("reg", RandomForestRegressor(
        n_estimators=400,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    ))
])

rf_reg.fit(X2_train, y2_train)

pred_rf = rf_reg.predict(X2_test)

# Evaluation
mse_rf = mean_squared_error(y2_test, pred_rf)
rmse_rf = np.sqrt(mse_rf)
mae_rf = mean_absolute_error(y2_test, pred_rf)
r2_rf = r2_score(y2_test, pred_rf)

print("Random Forest Performance:")
print("RMSE:", rmse_rf)
print("MAE:", mae_rf)
print("R²:", r2_rf)

Random Forest Performance:
RMSE: 22.52984510746301
MAE: 1.063593313749984
R²: 0.9968104575825544


In [39]:
''' Gradient Boosting Regressor '''
from sklearn.ensemble import GradientBoostingRegressor

gb_reg = Pipeline([
    ("preprocess", preprocess_2),
    ("reg", GradientBoostingRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])

gb_reg.fit(X2_train, y2_train)

pred_gb = gb_reg.predict(X2_test)

# Evaluation
mse_gb = mean_squared_error(y2_test, pred_gb)
rmse_gb = np.sqrt(mse_gb)
mae_gb = mean_absolute_error(y2_test, pred_gb)
r2_gb = r2_score(y2_test, pred_gb)

print("Gradient Boosting Performance:")
print("RMSE:", rmse_gb)
print("MAE:", mae_gb)
print("R²:", r2_gb)

Gradient Boosting Performance:
RMSE: 12.14251594461962
MAE: 3.9270404347290104
R²: 0.9990735343218046


In [40]:
# -----------------------------
# Ridge Regression
# -----------------------------
pred_ridge = grid_ridge_reg.best_estimator_.predict(X2_test)
r2_ridge = r2_score(y2_test, pred_ridge)

# -----------------------------
# Lasso Regression
# -----------------------------
pred_lasso = grid_lasso_reg.best_estimator_.predict(X2_test)
r2_lasso = r2_score(y2_test, pred_lasso)

# -----------------------------
# Elastic Net
# -----------------------------
pred_enet = grid_enet_reg.best_estimator_.predict(X2_test)
r2_enet = r2_score(y2_test, pred_enet)

# -----------------------------
# Create corrected table
# -----------------------------
results_df_corrected = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Ridge Regression",
        "Lasso Regression",
        "Elastic Net",
        "Random Forest",
        "Gradient Boosting"
    ],
    "RMSE": [
        rmse_lr,
        ridge_rmse,
        lasso_rmse,
        enet_rmse,
        rmse_rf,
        rmse_gb
    ],
    "R²": [
        r2_lr,
        r2_ridge,
        r2_lasso,
        r2_enet,
        r2_rf,
        r2_gb
    ]
})

results_df_corrected.style.highlight_min(subset=["RMSE"], color="lightgreen")

,Model,RMSE,R²
0,Linear Regression,74.447103,0.965174
1,Ridge Regression,75.232131,0.965173
2,Lasso Regression,75.377020,0.965166
3,Elastic Net,92.668515,0.945043
4,Random Forest,22.529845,0.996810
5,Gradient Boosting,12.142516,0.999074


In [41]:
''' Feature Importance for Ridge Regression (Coefficients) '''
# Extract the trained Ridge model from the pipeline
ridge_model = grid_ridge_reg.best_estimator_
ridge_regressor = ridge_model.named_steps["reg"]
preprocessor = ridge_model.named_steps["preprocess"]

# 1. Get feature names after preprocessing
# Numeric features (scaled)
numeric_features = numeric_cols_2

# Binary features (passed through)
binary_features = binary_cols_2

# Categorical features (OHE-expanded)
ohe = preprocessor.named_transformers_["cat"]
categorical_features = list(ohe.get_feature_names_out(categorical_cols_2))

# Combine all names in correct order:
all_features = numeric_features + categorical_features + binary_features

# 2. Extract coefficients
coefficients = ridge_regressor.coef_

# Sanity check
print("Number of transformed features:", len(all_features))
print("Number of coefficients:", len(coefficients))

# 3. Build importance DataFrame
ridge_importances = pd.DataFrame({
    "feature": all_features,
    "coefficient": coefficients,
    "abs_coeff": np.abs(coefficients)
}).sort_values(by="abs_coeff", ascending=False)

ridge_importances.head(20)

Number of transformed features: 80
Number of coefficients: 80


,feature,coefficient,abs_coeff
18,annual_medical_cost,385.353880,385.353880
67,network_tier_Platinum,121.965599,121.965599
65,network_tier_Bronze,-112.541795,112.541795
68,network_tier_Silver,-40.542891,40.542891
66,network_tier_Gold,31.119086,31.119086
13,deductible,10.013302,10.013302
79,had_major_procedure,-4.859100,4.859100
76,liver_disease,3.940068,3.940068
21,total_claims_paid,2.686523,2.686523
20,avg_claim_amount,-2.377212,2.377212
